# Isoglot 08: benign twins for the benign-share sweep

The first fine-tune closed the refusal gap but over-refused benign requests in Hausa, Yoruba and
Igbo (35-39%). Following the Lexsi team's suggestion, this writes a benign **twin** for each of the
120 harmful training seeds: a safe request on the same topic (how to stay safe, how it works, how to
get help) with a helpful answer, in Hausa, Yoruba and Igbo only. Twins whose English form is close
to any TukaBench prompt are dropped. CuratorKIT merges the twins with notebook 05's benign rows into
one export, `traindata/curated/benign_twins`, for notebook 09.

**Before running:** Accelerator **None**, Internet **on**, Secrets **`HF_TOKEN`** and
**`ANTHROPIC_API_KEY`** ticked. **Save Version -> Save & Run All (Commit)**. Hard cap: `CAP` below.


In [ ]:
%%capture
!pip install -q "isoglot[judge,mechanism,data] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os, json, collections
from pathlib import Path
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = s.get_secret("ANTHROPIC_API_KEY")
from huggingface_hub import HfApi, hf_hub_download, whoami
from huggingface_hub.errors import EntryNotFoundError
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
WORK = Path("/kaggle/working")
OUT = WORK / "traindata"
raw = Path(hf_hub_download(REPO, "traindata/raw/raw.jsonl", repo_type="dataset", local_dir=WORK))
print("first data:", sum(1 for _ in open(raw, encoding="utf-8")), "lines")


## The same 120 harmful seeds as notebook 05

In [ ]:
from isoglot.data import load_tukabench
from isoglot.traindata import decontaminate, load_seeds

seeds = load_seeds(n_harmful=120, n_edgy=90, n_plain=30, seed=0)
goals = sorted({x.metadata["goal_en"] for x in load_tukabench()})
seeds, _ = decontaminate(seeds, goals, threshold=0.5)
harmful = [x for x in seeds if x["kind"] == "harmful"]
print(len(harmful), "harmful seeds")


## Pilot: 3 twins, then project the cost

In [ ]:
from isoglot.claude import CLAUDE_PRICES, SpendMeter, client_from_env
from isoglot.traindata import generate_twins

MODEL = "claude-sonnet-5-5"
CAP = 3.00   # USD, hard stop for this notebook
meter = SpendMeter(CAP, *CLAUDE_PRICES[MODEL])
client = client_from_env()

rows, skipped = generate_twins(client, MODEL, meter, harmful[:3])
for r in rows:
    print(f"[{r['lang']}] twin of: {r['twin_of'][:60]}")
    print(f"    EN: {r['twin_en'][:100]}")
    print(f"    {r['instruction'][:90]}")
    print(f"    -> {r['output'][:140]}")
print("skipped:", skipped)
projected = meter.spent_usd / 3 * len(harmful)
print(f"pilot ${meter.spent_usd:.3f}; projected ${projected:.2f} for {len(harmful)} seeds (cap ${CAP:.2f})")
assert projected < CAP - meter.spent_usd, "projected cost is over the cap: stopping before the full run"


## All twins (resumable, 40 seeds per part)

In [ ]:
def remote(path): return path.relative_to(WORK).as_posix()
def publish(path):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=remote(path), repo_id=REPO, repo_type="dataset")

parts = []
for start in range(0, len(harmful), 40):
    path = OUT / "twins" / f"part-{start // 40:03d}.jsonl"; parts.append(path)
    if not path.exists():
        try:
            hf_hub_download(REPO, remote(path), repo_type="dataset", local_dir=WORK); print("restored", path.name)
        except EntryNotFoundError:
            pass
    if path.exists():
        continue
    rows, skipped = generate_twins(client, MODEL, meter, harmful[start:start + 40])
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows + [{"skipped": True, **x} for x in skipped]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    publish(path)
    print(f"{path.name}: {len(rows)} rows, {len(skipped)} skipped, spent ${meter.spent_usd:.2f}")
recs = [json.loads(l) for p in parts for l in open(p, encoding="utf-8")]
print(f"spent ${meter.spent_usd:.2f} | fallbacks {meter.fallbacks} | declined {meter.refusals}")


## Drop twins close to any TukaBench prompt, then curate with CuratorKIT

In [ ]:
from isoglot.traindata import jaccard, ngrams, curate

goal_grams = [ngrams(g) for g in goals]
def near_test(r): return max(jaccard(ngrams(r["twin_en"]), g) for g in goal_grams) >= 0.5
twins = [r for r in recs if not r.get("skipped")]
kept = [r for r in twins if not near_test(r)]
print(len(twins), "twin rows |", len(twins) - len(kept), "dropped as close to a TukaBench prompt |",
      collections.Counter(r["lang"] for r in kept))
twins_path = OUT / "twins" / "twins.jsonl"
with open(twins_path, "w", encoding="utf-8") as f:
    for r in kept:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
publish(twins_path)

curate([raw, twins_path], "benign", OUT / "curated", name="benign_twins").print_summary()
api.upload_folder(folder_path=str(OUT / "curated" / "benign_twins"), path_in_repo="traindata/curated/benign_twins",
                  repo_id=REPO, repo_type="dataset")
print("uploaded traindata/curated/benign_twins")
